# Regional inequality

This notebook is the page [Regional inequality](https://quarcs-lab.github.io/geo4bolivia/articles/inequality.html) of the [geo4bolivia documentation](https://quarcs-lab.github.io/geo4bolivia/). Run the cells from the top.

An inequality index puts in one number how far a set of values is from being all the same. This article asks three questions of Bolivia's municipalities:

1. **How unequal are they?** `geo.inequality()` computes the Gini index, Theil's index and the coefficient of variation, and draws the Lorenz curve.
2. **Is that changing?** With two periods the measures can be compared; with a long panel their trend can be tested.
3. **Is the inequality between departments or inside them?** `geo.theil_decomposition()` splits Theil's index in two.

It uses two variables on purpose: the **Sustainable Development Index**, a score between 0 and 100, and **population**, a count. An inequality index means different things for the two, and the difference is the main thing to take from this page.

In [ ]:
!pip install -q geo4bolivia

## The data

The dataset `sdg` has the Sustainable Development Index (`index_sdg`, IDS) and the census `population` of the 343 municipalities in two periods, stored as 2012 and 2024.

In [ ]:
import geo4bolivia as geo
import numpy as np
import pandas as pd

sdg = geo.load("sdg", groups=["department"])
cols = ["variable", "year", "count", "mean", "std", "min", "median", "max"]
geo.describe(sdg, variables=["index_sdg", "population"])[cols]

The two variables could hardly be more different. The index of 2024 lies between 47.3 and 77.3, around a mean of 60.1. Population in 2024 runs from 875 people to 1.6 million, with a mean (33,134) far above the median (12,296).

## Inequality between municipalities, not between people
Every measure on this page treats each municipality as one observation, whatever its size: the municipality of 875 people counts as much as the one of 1.6 million. The results describe how different *municipalities* are. They are not measures of inequality between people or households, which would need data on people.

## 1. Three measures

In [ ]:
res = geo.inequality(sdg, "index_sdg", measures=("gini", "theil", "cv"))
res

All three measures are 0 when every unit has the same value and grow as the values spread out. They summarise the spread differently:

| Measure | What it is | Range | Reacts most to |
| --- | --- | --- | --- |
| **Gini index** (`"gini"`) | the average difference between two units, over all pairs, divided by twice the mean | 0 to 1 | differences in the middle of the distribution, where most units are |
| **Theil index** (`"theil"`) | the average of (value ÷ mean) × ln(value ÷ mean) | 0 to ln(number of units) | high values more than low ones; it splits exactly into groups (section 6) |
| **Coefficient of variation** (`"cv"`) | the standard deviation divided by the mean | 0 upwards | the highest values, because deviations are squared |

The Theil index takes logarithms, so it needs strictly positive values; with zeros in the data, use `measures=("gini", "cv")`.

All three are **relative** measures: they compare the values with their mean. Doubling every value leaves them unchanged. *Adding* the same amount to every value does not: it lowers all three, because the same gaps are then smaller relative to a higher mean. Keep this in mind for section 3.

`.gt` is the table of the result, and `.df` has the same numbers, one row per year:

In [ ]:
res.gt

By all three measures the index is more evenly spread in 2024: the Gini index fell from 0.065 to 0.049, the Theil index from 0.0066 to 0.0038 and the coefficient of variation from 0.116 to 0.088. With two periods that is all that can be said: the values can be compared, but there is no test of a trend, which needs three or more years (`res.notes` says so).

## 2. The Lorenz curve

The Lorenz curve is the picture behind the Gini index. Sort the municipalities from the lowest value to the highest. The curve shows, for each share of municipalities counted from the bottom (horizontal axis), the share of the total of the variable that they hold (vertical axis).

In [ ]:
res.fig_lorenz

How to read it:

- The dashed diagonal is **equality**: if every municipality had the same value, the lowest 30% of them would hold 30% of the total.
- The curve can only lie on or below the diagonal. The further it sags, the more unequal the values.
- The **Gini index** is the area between the diagonal and the curve, as a share of the whole triangle under the diagonal.
- When two curves do not cross, the higher one is the more equal by any of the three measures. When they cross, the measures can disagree, because each weighs the parts of the distribution differently.

`.lorenz_share(year, share)` reads a point off the curve: the share of the total held by the lowest `share` of units. `.lorenz` is the table behind the figure.

In [ ]:
for year in [2012, 2024]:
    half = res.lorenz_share(year, 0.5)
    top = 1 - res.lorenz_share(year, 0.9)
    print(year, "| lowest half:", round(half, 3), "| highest tenth:", round(top, 3))

For the index the curves almost lie on the diagonal. The half of the municipalities with the lowest index holds 45.4% of the sum of the index in 2012 and 46.5% in 2024, where equality would give 50%; the tenth with the highest index holds about 12%, where equality would give 10%. The curve of 2024 lies above the curve of 2012 along its whole length, which is why all three measures agree that the spread fell.

## 3. What an index of a score can and cannot say

Now the same call on population.

In [ ]:
pop = geo.inequality(sdg, "population", measures=("gini", "theil", "cv"))
pop.fig_lorenz

In [ ]:
print(pop.df.round(3))
half, top = pop.lorenz_share(2024, 0.5), 1 - pop.lorenz_share(2024, 0.9)
print("lowest half:", round(half, 3), "| highest tenth:", round(top, 3))

The Gini index of population is 0.70 in both censuses. The half of the municipalities with the fewest people holds 9.8% of the population in 2024, and the tenth with the most people holds 64%.

It is tempting to conclude that population is fourteen times more unequal than development (0.70 against 0.05). That comparison means nothing, for three reasons.

**A count has a total that is shared out; a score does not.** For population the Lorenz curve can be read literally: 64% of the people live in a tenth of the municipalities. For the index, "the total" is the sum of 343 scores, which is not an amount of anything and which no one holds. The curve and the Gini index of a score still describe how spread out the scores are, relative to their mean, but the reading "the lowest half holds 46% of the total" is a statement about arithmetic, not about who has what.

**A bounded score cannot be very unequal.** Population can differ by a factor of more than a thousand between two municipalities. An index that lies between 47 and 77 cannot: its highest value is 1.6 times its lowest. Its Gini index is bound to be small whatever happens in the country. A small value is a property of the scale, so compare the index with itself, between years or between groups of municipalities, not with other variables.

**The direction of the change depends on how the score is written.** A score can be turned around: an index of what has been achieved, or an index of what is missing. The two hold the same information, but relative measures do not treat them alike. Multidimensional poverty (`sdg1_2_mpi`, from 0 to 1, lower is better) shows it clearly. It fell in every municipality between the two censuses.

In [ ]:
mpi = geo.load("sdg01", variables="sdg1_2_mpi")
mpi["not_poor"] = 1 - mpi["sdg1_2_mpi"]
rows = []
for variable in ["sdg1_2_mpi", "not_poor"]:
    gini = geo.inequality(mpi, variable, measures="gini", lorenz=False).df
    spread = mpi.groupby("year")[variable].agg(["mean", "std"]).reset_index()
    rows.append(gini.merge(spread, on="year").assign(variable=variable))
pd.concat(rows)[["variable", "year", "mean", "std", "gini"]].round(3)

The Gini index of poverty **rose**, from 0.254 to 0.367, while the Gini index of its mirror image, one minus the poverty index, **fell**, from 0.078 to 0.025. Both describe the same municipalities in the same years. The standard deviation, which is identical for the two, fell from 0.106 to 0.044: the municipalities are closer to each other in index points. But mean poverty fell even faster, so the remaining gaps are larger *relative to the mean* of poverty, and smaller relative to the mean of its mirror image.

Neither Gini is wrong. "Poverty became more concentrated in a few municipalities" and "municipalities became more alike in how much of their population is out of poverty" are both true. The lesson is to report the mean and the standard deviation next to any relative measure of a score, and to say which way the score runs.

The Sustainable Development Index runs upwards (100 is best), and its mean rose by ten points. How much of the fall of its inequality is the higher mean?

In [ ]:
spread = sdg.groupby("year")["index_sdg"].agg(["mean", "std"])
spread["cv"] = spread["std"] / spread["mean"]
print(spread.round(3))
print(round(spread.loc[2012, "std"] / spread.loc[2024, "mean"], 3))

The standard deviation fell by about a tenth, from 5.82 to 5.26 points, and the coefficient of variation by about a quarter, from 0.116 to 0.088. Had the gaps in points stayed exactly as in 2012 while the mean rose to 60.1, the coefficient of variation would still have fallen, to 0.097. So about two thirds of its fall comes from the higher level and one third from municipalities moving closer together in points. Both parts are real, and they are different facts: "everyone rose by a similar amount" and "the lowest rose more". The [convergence article](https://quarcs-lab.github.io/geo4bolivia/articles/convergence.html) looks at the second directly.

## 4. A long panel: testing the trend

With three or more years, `geo.inequality()` also estimates the **trend** of each measure: it regresses the logarithm of the measure on the year, so the slope is the average proportional change per year. The dataset `pop` has a gridded population estimate (WorldPop) for the 339 municipalities of the 2015 map (`map339`) in every year from 2001 to 2020.

In [ ]:
long = geo.inequality("pop", "pop", measures=("gini", "theil", "cv"))
long

In [ ]:
long.summary[["measure", "first", "last", "change", "slope", "se", "pvalue"]]

The Gini index of municipal population rose from 0.691 in 2001 to 0.710 in 2020. Its slope is 0.0014: the index grew by 0.14% a year on average, and the trend is statistically significant (the p-value is far below 0.01). The Theil index rose at the same rate (0.14% a year). The coefficient of variation hardly moved: its slope, 0.02% a year, is a sixth of the others, although with twenty smooth values it is still distinguishable from zero (p = 0.006).

So the measures differ in size, and the table in section 1 says why they can. The Gini index reacts to the middle of the distribution and the coefficient of variation to its very top. Read together, they say that population became somewhat more concentrated among the municipalities in general, while the distance between the very largest ones and the average barely changed.

Three cautions about this result:

- The series is a modelled estimate that changes smoothly from year to year, which is why the standard error of the Gini trend is so small. The test says that a straight line fits these twenty values closely; it does not have twenty independent measurements behind it.
- The 339 municipalities are those of the 2015 map, on which the population was computed; the four units created since have no values, and six municipalities count their 2015 territory (`geo.info("pop")` explains it). The result describes them.
- `first=` and `last=` restrict the years, for instance `geo.inequality("pop", "pop", first=2012)`. A trend over a different window can differ.

With more than four years the Lorenz figure draws only the first and the last year; `.lorenz` keeps all of them.

## 5. The spatial Gini: do neighbours differ less?

The Gini index is an average over all pairs of units. `spatial=True` separates the pairs into two kinds, **neighbours** and all the others, and reports the part of the Gini index that comes from pairs of neighbours (`gini_spatial`). A permutation test then asks whether the values are arranged in space so that neighbours differ less than other pairs: it shuffles the values over the map and compares. The neighbours are those of the spatial weights, queen contiguity by default ([Maps and spatial dependence](https://quarcs-lab.github.io/geo4bolivia/articles/spatial-dependence.html) explains them); `w=` takes others.

In [ ]:
sp = geo.inequality(sdg, "index_sdg", spatial=True, permutations=999)
linked = geo.weights("map343").pct_nonzero
print(sp.df.round(5))
print((sp.df["gini_spatial"] / sp.df["gini"]).round(4).tolist(), round(linked, 2))

In 2024, pairs of neighbouring municipalities account for 0.00057 of the Gini index of 0.0493, which is 1.2% of it. That share is small mostly because few pairs are neighbours: 1.6% of all pairs of municipalities. The informative part is the comparison of the two percentages and the test. Neighbours are 1.6% of the pairs but contribute 1.2% of the inequality, so two neighbouring municipalities differ less than two municipalities taken at random, and none of the 999 shuffles gave a pattern as strong (`gini_spatial_p` = 0.001). The same holds in 2012.

This is the spatial clustering of the index, seen from the side of inequality: most of the differences are between places that are far apart. It says nothing about why neighbours are alike. With `spatial=True` every measure refers to the units that have data in every year and a boundary; `res.notes` reports any that were left out.

## 6. Between and within groups

Is the inequality between municipalities a matter of rich and poor *departments*, or of differences inside each department? Theil's index can be split exactly in two parts that add up to it:

- **between**: the inequality there would be if every municipality had the average of its group;
- **within**: the inequality among the municipalities of the same group, averaged over the groups.

`by=` names the groups: a column of the table, or a coarser scale that the units nest in.

In [ ]:
dep = geo.theil_decomposition(sdg, "index_sdg", by="department",
                              permutations=999)
dep

In [ ]:
dep.df.round(4)

Differences between the averages of the nine departments are 20% of the inequality between municipalities in 2012 and 15% in 2024. Most of it, 85% in 2024, is inside departments. Both parts fell, the between part faster (from 0.0014 to 0.0006) than the within part (from 0.0053 to 0.0032).

### Is the between part more than chance?

Any grouping gives a between part above zero, even one drawn from a hat, because group averages always differ a little. `permutations=` tests the grouping: it reassigns the municipalities to groups of the same sizes at random, many times, and counts how often the random groups give a between part as large as the real ones (`p_between`). Here none of 999 did (p = 0.001): the departments separate municipalities with higher and lower values more than an arbitrary grouping would, although they hold only 15% of the inequality.

### More groups, a larger between share

The same decomposition by province gives a very different number.

In [ ]:
prov = geo.theil_decomposition(sdg, "index_sdg", by="province",
                               permutations=999)
prov.df.round(4)

Between the 113 provinces lies 66% of the inequality in 2024. That does not make provinces four times as important as departments. The between share **always grows when the groups are made finer**, for a mechanical reason: with fewer units in each group there is less left to differ inside it. At the limit, with every municipality its own group, the between share is 100%. The provinces have three municipalities each on average, and 16 of them have only one, so they contribute nothing to the within part by construction.

How large is the mechanical part? Shuffle the municipalities among the groups, keeping the sizes of the groups, and see what share random groups give. This is what the permutation test does internally.

In [ ]:
units = geo.units("map343")[["geo_id", "department_id", "province_id"]]
rng = np.random.default_rng(1)
rows = []
for by, real in [("department_id", dep), ("province_id", prov)]:
    shares = []
    for _ in range(50):
        shuffled = units.assign(group=rng.permutation(units[by]))
        table = sdg.merge(shuffled[["geo_id", "group"]], on="geo_id")
        random = geo.theil_decomposition(table, "index_sdg", by="group")
        shares.append(random.df["between_share"].iloc[-1])
    rows.append({"groups": by, "number": units[by].nunique(),
                 "between share": real.df["between_share"].iloc[-1],
                 "random groups (mean)": np.mean(shares)})
pd.DataFrame(rows).round(3)

Nine random groups of the sizes of the departments hold about 2% of the inequality; the real departments hold 15%. A hundred and thirteen random groups of the sizes of the provinces hold about a third; the real provinces hold 66%. Both real groupings are well above their random counterpart, and that comparison, not the raw share, is what can be set side by side. Compare between shares only for groupings with a similar number of groups, or against the random benchmark.

### The same split for population

In [ ]:
people = geo.theil_decomposition(sdg, "population", by="department",
                                 permutations=999)
people.df.round(3)

For population the departments hold 6 to 7% of the inequality between municipalities, and random groupings give as much or more in most of the shuffles (p = 0.83 in 2012 and 0.69 in 2024): the between part is not distinguishable from chance. The large differences in population are between the municipalities inside each department, not between the departments' averages.

## In short

- `geo.inequality()` gives the Gini index, Theil's index and the coefficient of variation in each year. They are relative measures and react to different parts of the distribution, so they can disagree, as they nearly do for population between 2001 and 2020 (the Gini index rose six times faster than the coefficient of variation).
- The Lorenz curve shows the whole distribution: the further it sags below the diagonal, the more unequal the values. `.lorenz_share()` reads points off it.
- The Sustainable Development Index is more evenly spread across municipalities in 2024 than in 2012 by all three measures (Gini 0.065 to 0.049). About two thirds of the fall of its coefficient of variation comes from the higher mean, one third from smaller gaps in points.
- An inequality index of a bounded score is small by construction, has no "total" that is shared out, and can change direction when the score is turned around. Compare it with itself, and report the mean and the standard deviation with it.
- With two periods the measures can be compared; a trend test needs three or more years.
- The spatial Gini shows that neighbouring municipalities differ less than others.
- Differences between departments are 15% of the inequality of the index in 2024, more than random groups would give. Between shares grow with the number of groups, so compare them against a random benchmark.
- The units are municipalities, each counted once. None of this measures inequality between people, and none of it explains where the differences come from.

## Where next

- **[Convergence: β, σ and clubs](https://quarcs-lab.github.io/geo4bolivia/articles/convergence.html)**: did the municipalities that started lower rise faster, and does dispersion fall over time?
- **[Transitions and mobility](https://quarcs-lab.github.io/geo4bolivia/articles/transitions.html)**: who moved up and down while the spread changed.
- **[Maps and spatial dependence](https://quarcs-lab.github.io/geo4bolivia/articles/spatial-dependence.html)**: spatial weights and the clustering behind the spatial Gini.
- **[Exploring the data](https://quarcs-lab.github.io/geo4bolivia/articles/exploring.html)**: distributions and summaries before any index.
- **[Analyze](https://quarcs-lab.github.io/geo4bolivia/analyze.html)**: all the families on one question.
- Reference: [`inequality`](https://quarcs-lab.github.io/geo4bolivia/reference/inequality.html), [`theil_decomposition`](https://quarcs-lab.github.io/geo4bolivia/reference/theil_decomposition.html).